# Ch.5 — Monitoring & Observability (Azure Cloud Supplement)

> **Goal:** Integrate Flask app with Azure Application Insights for cloud-native monitoring, send metrics with OpenTelemetry, and create Azure Monitor workbooks.
>
> **Tech stack:** Azure Application Insights, OpenTelemetry, Azure Monitor, Azure CLI
>
> **What you'll build:** A production Flask app sending metrics, logs, and traces to Azure — with alerting and dashboarding in Azure Monitor.
>
> ⚠️ **Azure credentials required:** This notebook requires an active Azure subscription. Credentials are stripped by pre-push hook.

In [ ]:
# Cell 1: Azure credentials

# ⚠️ WARNING: These credentials are STRIPPED by pre-push hook
# Never commit real credentials to version control

AZURE_SUBSCRIPTION_ID = ""  # Your Azure subscription ID
AZURE_RESOURCE_GROUP = "monitoring-demo-rg"  # Resource group name
AZURE_LOCATION = "eastus"  # Azure region
AZURE_APP_INSIGHTS_NAME = "flask-app-insights"  # Application Insights instance name

# Validate credentials
if not AZURE_SUBSCRIPTION_ID:
    print("❌ AZURE_SUBSCRIPTION_ID is empty")
    print("\n📝 To get your subscription ID:")
    print("  az account list --output table")
    print("\nOr use Azure Portal:")
    print("  Portal → Subscriptions → Copy 'Subscription ID'")
else:
    print("✅ Azure credentials configured")
    print(f"  Subscription ID: {AZURE_SUBSCRIPTION_ID[:8]}...")
    print(f"  Resource Group: {AZURE_RESOURCE_GROUP}")
    print(f"  Location: {AZURE_LOCATION}")
    print(f"  App Insights: {AZURE_APP_INSIGHTS_NAME}")

print("\n⚠️ Reminder: These credentials will be stripped before git push")

In [ ]:
# Cell 2: Azure Application Insights (APM alternative)

# Azure Application Insights is a cloud-native APM (Application Performance Monitoring) service
# It provides metrics, logs, and distributed tracing without running your own Prometheus/Grafana

import subprocess
import json

def run_az_command(command):
    """Run Azure CLI command and return JSON output"""
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f"❌ Command failed: {result.stderr}")
        return None
    return json.loads(result.stdout) if result.stdout else None

# Check Azure CLI installation
print("🔍 Checking Azure CLI installation...")
az_version = subprocess.run(['az', '--version'], capture_output=True, text=True)
if az_version.returncode != 0:
    print("❌ Azure CLI not installed")
    print("\n📦 Install Azure CLI:")
    print("  Windows: winget install Microsoft.AzureCLI")
    print("  macOS: brew install azure-cli")
    print("  Linux: curl -sL https://aka.ms/InstallAzureCLIDeb | sudo bash")
else:
    print("✅ Azure CLI installed")
    print(az_version.stdout.split('\n')[0])  # Show version

# Login to Azure
print("\n🔐 Logging in to Azure...")
print("  Running: az login")
print("  (This will open a browser window for authentication)")
subprocess.run(['az', 'login'], check=True)

# Set active subscription
if AZURE_SUBSCRIPTION_ID:
    print(f"\n🎯 Setting active subscription to {AZURE_SUBSCRIPTION_ID[:8]}...")
    subprocess.run(['az', 'account', 'set', '--subscription', AZURE_SUBSCRIPTION_ID], check=True)
    print("✅ Subscription set")

# Create resource group
print(f"\n📦 Creating resource group '{AZURE_RESOURCE_GROUP}' in {AZURE_LOCATION}...")
rg_result = run_az_command(
    f'az group create --name {AZURE_RESOURCE_GROUP} --location {AZURE_LOCATION}'
)
if rg_result:
    print(f"✅ Resource group created: {rg_result['name']}")

# Create Application Insights instance
print(f"\n📊 Creating Application Insights instance '{AZURE_APP_INSIGHTS_NAME}'...")
appinsights_result = run_az_command(
    f'az monitor app-insights component create '
    f'--app {AZURE_APP_INSIGHTS_NAME} '
    f'--location {AZURE_LOCATION} '
    f'--resource-group {AZURE_RESOURCE_GROUP} '
    f'--application-type web'
)

if appinsights_result:
    instrumentation_key = appinsights_result['instrumentationKey']
    connection_string = appinsights_result['connectionString']
    
    print("✅ Application Insights created")
    print(f"  Instrumentation Key: {instrumentation_key[:8]}...")
    print(f"  Connection String: {connection_string[:50]}...")
    
    # Save to file for next cell
    with open('azure_app_insights_config.json', 'w') as f:
        json.dump({
            'instrumentation_key': instrumentation_key,
            'connection_string': connection_string,
            'resource_group': AZURE_RESOURCE_GROUP,
            'app_insights_name': AZURE_APP_INSIGHTS_NAME
        }, f, indent=2)
    
    print("\n✅ Configuration saved to azure_app_insights_config.json")
else:
    print("❌ Failed to create Application Insights instance")

print("\n🔗 View Application Insights in Azure Portal:")
print(f"  https://portal.azure.com/#@/resource/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}/providers/microsoft.insights/components/{AZURE_APP_INSIGHTS_NAME}")

In [ ]:
# Cell 3: Instrument Flask app with OpenTelemetry

# OpenTelemetry is the CNCF standard for observability instrumentation
# It provides a unified API for metrics, logs, and traces

# Install OpenTelemetry packages
print("📦 Installing OpenTelemetry packages...")
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-instrumentation-flask \
             opentelemetry-exporter-otlp azure-monitor-opentelemetry-exporter

# Load Application Insights configuration
try:
    with open('azure_app_insights_config.json', 'r') as f:
        config = json.load(f)
    connection_string = config['connection_string']
    print("\n✅ Loaded Application Insights configuration")
except FileNotFoundError:
    print("❌ Configuration file not found. Run Cell 2 first.")
    connection_string = None

# Create instrumented Flask app
if connection_string:
    app_code = f'''
from flask import Flask, request
from opentelemetry import trace, metrics
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.instrumentation.flask import FlaskInstrumentor
from azure.monitor.opentelemetry.exporter import AzureMonitorTraceExporter, AzureMonitorMetricExporter
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
import time
import random

# Configure OpenTelemetry with Azure Monitor exporter
trace.set_tracer_provider(TracerProvider())
tracer = trace.get_tracer(__name__)

# Azure Monitor exporters
trace_exporter = AzureMonitorTraceExporter(
    connection_string="{connection_string}"
)
span_processor = BatchSpanProcessor(trace_exporter)
trace.get_tracer_provider().add_span_processor(span_processor)

metric_exporter = AzureMonitorMetricExporter(
    connection_string="{connection_string}"
)
metric_reader = PeriodicExportingMetricReader(metric_exporter)
metrics.set_meter_provider(MeterProvider(metric_readers=[metric_reader]))
meter = metrics.get_meter(__name__)

# Create custom metrics
request_counter = meter.create_counter(
    "http.server.requests",
    description="Total HTTP requests",
    unit="1"
)

request_duration = meter.create_histogram(
    "http.server.duration",
    description="HTTP request duration",
    unit="ms"
)

app = Flask(__name__)

# Auto-instrument Flask (captures all routes automatically)
FlaskInstrumentor().instrument_app(app)

@app.route('/health')
def health():
    return {{'status': 'ok', 'service': 'payment-api'}}, 200

@app.route('/api/payment', methods=['POST'])
def payment():
    with tracer.start_as_current_span("process_payment") as span:
        start_time = time.time()
        
        # Simulate processing
        time.sleep(random.uniform(0.05, 0.3))
        
        # Add custom span attributes
        span.set_attribute("payment.amount", 100)
        span.set_attribute("payment.currency", "USD")
        
        # Record metrics
        duration_ms = (time.time() - start_time) * 1000
        request_counter.add(1, {{"route": "/api/payment", "status": "200"}})
        request_duration.record(duration_ms, {{"route": "/api/payment"}})
        
        if random.random() < 0.05:
            span.set_attribute("error", True)
            return {{'error': 'Payment gateway timeout'}}, 503
        
        return {{'status': 'success', 'transaction_id': random.randint(1000, 9999)}}, 200

@app.route('/api/refund', methods=['POST'])
def refund():
    with tracer.start_as_current_span("process_refund") as span:
        start_time = time.time()
        
        time.sleep(random.uniform(0.2, 0.8))
        
        span.set_attribute("refund.transaction_id", 1234)
        
        duration_ms = (time.time() - start_time) * 1000
        request_counter.add(1, {{"route": "/api/refund", "status": "200"}})
        request_duration.record(duration_ms, {{"route": "/api/refund"}})
        
        if random.random() < 0.10:
            span.set_attribute("error", True)
            return {{'error': 'Refund processing failed'}}, 500
        
        return {{'status': 'refunded', 'transaction_id': random.randint(1000, 9999)}}, 200

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''
    
    with open('app_azure.py', 'w') as f:
        f.write(app_code)
    
    print("\n✅ Flask app instrumented with OpenTelemetry")
    print("✅ Saved to app_azure.py")
    print("\n📊 OpenTelemetry features:")
    print("  - Auto-instrumentation: All Flask routes tracked automatically")
    print("  - Distributed tracing: Each request gets a unique trace ID")
    print("  - Custom metrics: http.server.requests, http.server.duration")
    print("  - Span attributes: Custom metadata (payment amount, transaction ID)")
    print("\n🚀 Run the app:")
    print("  python app_azure.py")
    print("\n💡 All telemetry will be sent to Azure Application Insights")

In [ ]:
# Cell 4: Send metrics to Application Insights

# Start the Flask app in the background and generate traffic

import subprocess
import time
import requests
import threading

def run_flask_app():
    """Run Flask app in background"""
    subprocess.run(['python', 'app_azure.py'], check=True)

# Start Flask app in a separate thread
print("🚀 Starting Flask app in background...")
flask_thread = threading.Thread(target=run_flask_app, daemon=True)
flask_thread.start()

# Wait for Flask to start
print("⏳ Waiting for Flask to start...")
time.sleep(5)

# Check if Flask is running
try:
    response = requests.get('http://localhost:5000/health', timeout=5)
    if response.status_code == 200:
        print("✅ Flask app is running")
    else:
        print(f"⚠️ Flask returned status {response.status_code}")
except Exception as e:
    print(f"❌ Cannot reach Flask app: {e}")
    print("\nTry running manually:")
    print("  python app_azure.py")

# Generate traffic
print("\n🚀 Generating traffic to populate Application Insights...")

def send_requests(num_requests=100):
    """Send synthetic requests to Flask app"""
    urls = [
        ('http://localhost:5000/health', 'GET', None, 0.6),
        ('http://localhost:5000/api/payment', 'POST', {'amount': 100}, 0.3),
        ('http://localhost:5000/api/refund', 'POST', {'transaction_id': 1234}, 0.1),
    ]
    
    status_counts = {}
    
    for i in range(num_requests):
        # Weighted random selection
        import random
        rand = random.random()
        cumulative = 0
        for url, method, json_data, weight in urls:
            cumulative += weight
            if rand < cumulative:
                try:
                    if method == 'POST':
                        resp = requests.post(url, json=json_data, timeout=5)
                    else:
                        resp = requests.get(url, timeout=5)
                    status_counts[resp.status_code] = status_counts.get(resp.status_code, 0) + 1
                except Exception as e:
                    status_counts['error'] = status_counts.get('error', 0) + 1
                break
        
        # Progress
        if (i + 1) % 20 == 0:
            print(f"  Progress: {i + 1}/{num_requests} requests sent")
        
        # Rate limiting
        time.sleep(0.1)
    
    return status_counts

status_counts = send_requests(100)

print("\n✅ Traffic generation complete")
print("\n📊 Response status distribution:")
for status, count in sorted(status_counts.items()):
    percentage = (count / 100) * 100
    print(f"  {status}: {count} ({percentage:.1f}%)")

print("\n⏳ Telemetry is being sent to Azure Application Insights...")
print("   (It may take 1-2 minutes for data to appear in the portal)")

print("\n🔗 View telemetry in Azure Portal:")
print(f"  https://portal.azure.com/#@/resource/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}/providers/microsoft.insights/components/{AZURE_APP_INSIGHTS_NAME}/overview")

print("\n📊 What to look for in Application Insights:")
print("  - Overview → Server response time graph (should show ~0.2s)")
print("  - Overview → Server requests graph (should show ~100 requests)")
print("  - Failures → List of 5xx errors with stack traces")
print("  - Performance → Slowest operations (refund should be slower than payment)")
print("  - Transaction search → Individual request traces with full context")

In [ ]:
# Cell 5: Create Azure Monitor workbook (dashboard)

# Azure Monitor Workbooks are similar to Grafana dashboards
# They provide interactive visualizations with KQL (Kusto Query Language)

# Create a workbook template
workbook_template = {
    "version": "Notebook/1.0",
    "items": [
        {
            "type": 1,
            "content": {
                "json": "# Flask App Monitoring Dashboard\n\nReal-time metrics, logs, and traces from Application Insights"
            },
            "name": "text - 0"
        },
        {
            "type": 3,
            "content": {
                "version": "KqlItem/1.0",
                "query": "requests\n| summarize RequestCount = count() by bin(timestamp, 1m), name\n| render timechart",
                "size": 0,
                "title": "Request Rate (requests/min)",
                "queryType": 0,
                "resourceType": "microsoft.insights/components"
            },
            "name": "query - 1"
        },
        {
            "type": 3,
            "content": {
                "version": "KqlItem/1.0",
                "query": "requests\n| summarize p50 = percentile(duration, 50), p95 = percentile(duration, 95) by bin(timestamp, 1m), name\n| render timechart",
                "size": 0,
                "title": "Latency Percentiles (p50, p95)",
                "queryType": 0,
                "resourceType": "microsoft.insights/components"
            },
            "name": "query - 2"
        },
        {
            "type": 3,
            "content": {
                "version": "KqlItem/1.0",
                "query": "requests\n| where resultCode >= 500\n| summarize ErrorCount = count() by bin(timestamp, 1m), name\n| render timechart",
                "size": 0,
                "title": "Error Rate (5xx errors)",
                "queryType": 0,
                "resourceType": "microsoft.insights/components"
            },
            "name": "query - 3"
        },
        {
            "type": 3,
            "content": {
                "version": "KqlItem/1.0",
                "query": "requests\n| summarize TotalRequests = count(), AvgDuration = avg(duration), ErrorCount = countif(resultCode >= 500) by name\n| extend ErrorRate = ErrorCount * 100.0 / TotalRequests\n| project name, TotalRequests, AvgDuration, ErrorRate\n| order by TotalRequests desc",
                "size": 0,
                "title": "Route Summary Table",
                "queryType": 0,
                "resourceType": "microsoft.insights/components"
            },
            "name": "query - 4"
        }
    ],
    "styleSettings": {},
    "$schema": "https://github.com/Microsoft/Application-Insights-Workbooks/blob/master/schema/workbook.json"
}

with open('azure_workbook_template.json', 'w') as f:
    json.dump(workbook_template, f, indent=2)

print("✅ Azure Monitor Workbook template saved to azure_workbook_template.json")

print("\n📊 Workbook queries (KQL - Kusto Query Language):")
print("="*60)
print("\n1. Request Rate (requests/min):")
print("   requests")
print("   | summarize RequestCount = count() by bin(timestamp, 1m), name")
print("   | render timechart")

print("\n2. Latency Percentiles (p50, p95):")
print("   requests")
print("   | summarize p50 = percentile(duration, 50), p95 = percentile(duration, 95)")
print("   | render timechart")

print("\n3. Error Rate (5xx errors):")
print("   requests")
print("   | where resultCode >= 500")
print("   | summarize ErrorCount = count() by bin(timestamp, 1m)")
print("   | render timechart")

print("\n4. Route Summary Table:")
print("   requests")
print("   | summarize TotalRequests = count(), AvgDuration = avg(duration)")
print("   | extend ErrorRate = ErrorCount * 100.0 / TotalRequests")

print("\n📝 To create the workbook in Azure Portal:")
print("  1. Open Application Insights → Workbooks → New")
print("  2. Click 'Advanced Editor' (</> icon)")
print("  3. Paste contents of azure_workbook_template.json")
print("  4. Click 'Apply' → 'Done Editing' → Save")

print("\n🔗 Or navigate directly:")
print(f"  https://portal.azure.com/#@/resource/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}/providers/microsoft.insights/components/{AZURE_APP_INSIGHTS_NAME}/workbooks")

print("\n✅ Workbook will auto-refresh and show live data")

In [ ]:
# Cell 6: Set up alerts (email/SMS on high error rate)

# Azure Monitor alerts use KQL queries to detect anomalies
# and send notifications via email, SMS, webhook, or Azure Action Groups

# Create an alert rule configuration
alert_rule = {
    "name": "high-error-rate-alert",
    "description": "Alert when error rate exceeds 5% for 5 minutes",
    "severity": 2,  # 0=Critical, 1=Error, 2=Warning, 3=Info
    "enabled": True,
    "evaluationFrequency": "PT5M",  # Every 5 minutes
    "windowSize": "PT5M",  # Look back 5 minutes
    "criteria": {
        "allOf": [
            {
                "query": "requests | where resultCode >= 500 | summarize ErrorCount = count(), TotalCount = countif(true) | extend ErrorRate = ErrorCount * 100.0 / TotalCount | where ErrorRate > 5",
                "timeAggregation": "Count",
                "operator": "GreaterThan",
                "threshold": 0,
                "failingPeriods": {
                    "numberOfEvaluationPeriods": 1,
                    "minFailingPeriodsToAlert": 1
                }
            }
        ]
    }
}

with open('azure_alert_rule.json', 'w') as f:
    json.dump(alert_rule, f, indent=2)

print("✅ Alert rule configuration saved to azure_alert_rule.json")

print("\n🚨 Configured alerts:")
print("="*60)
print("\n1. High Error Rate Alert")
print("   - Condition: Error rate > 5% for 5 minutes")
print("   - Severity: Warning (2)")
print("   - Evaluation: Every 5 minutes")
print("   - Query: requests | where resultCode >= 500")

print("\n📝 To create the alert in Azure Portal:")
print("  1. Open Application Insights → Alerts → New alert rule")
print("  2. Scope: Select your Application Insights resource")
print("  3. Condition: Custom log search")
print("  4. Query:")
print("     requests")
print("     | where resultCode >= 500")
print("     | summarize ErrorCount = count(), TotalCount = countif(true)")
print("     | extend ErrorRate = ErrorCount * 100.0 / TotalCount")
print("     | where ErrorRate > 5")
print("  5. Alert logic: Threshold = 0, Operator = Greater than")
print("  6. Evaluation frequency: 5 minutes, Period: 5 minutes")
print("  7. Actions: Create Action Group (email/SMS/webhook)")

print("\n📧 Create Action Group for notifications:")
print("  1. Alerts → Action groups → Create action group")
print("  2. Basics: Name = 'devops-alerts', Resource group = monitoring-demo-rg")
print("  3. Notifications:")
print("     - Type: Email/SMS/Push/Voice")
print("     - Name: 'Email DevOps Team'")
print("     - Email: your-email@example.com")
print("  4. Actions (optional):")
print("     - Webhook: POST to Slack/PagerDuty/custom endpoint")
print("     - Azure Function: Run remediation script")
print("  5. Create action group → Link to alert rule")

print("\n✅ When the alert fires, you'll receive:")
print("  - Email notification with alert details")
print("  - SMS (if configured)")
print("  - Webhook POST to Slack/PagerDuty")
print("  - Azure Portal notification")

print("\n🧪 Test the alert:")
print("  1. Modify app_azure.py to increase error rate:")
print("     if random.random() < 0.20:  # 20% error rate")
print("  2. Generate traffic (run Cell 4)")
print("  3. Wait 5-10 minutes for alert to fire")
print("  4. Check email for alert notification")

print("\n🔗 View alerts in Azure Portal:")
print(f"  https://portal.azure.com/#@/resource/subscriptions/{AZURE_SUBSCRIPTION_ID}/resourceGroups/{AZURE_RESOURCE_GROUP}/providers/microsoft.insights/components/{AZURE_APP_INSIGHTS_NAME}/alertsV2")

print("\n🎯 Summary — What You Built:")
print("="*60)
print("✅ Azure Application Insights instance provisioned")
print("✅ Flask app instrumented with OpenTelemetry")
print("✅ Metrics, logs, and traces sent to Azure Monitor")
print("✅ Azure Monitor Workbook with custom dashboards")
print("✅ Alert rules for high error rate with email notifications")
print("\n🚀 Next Steps:")
print("  - Add distributed tracing across microservices")
print("  - Integrate with Azure DevOps CI/CD pipelines")
print("  - Set up log analytics for structured log querying")
print("  - Configure auto-scaling based on metrics (Azure App Service)")
print("\n🧹 Cleanup (to avoid Azure charges):")
print(f"  az group delete --name {AZURE_RESOURCE_GROUP} --yes --no-wait")